# Cape Jones — Lighting Gag Reel (Colab T4 GPU)

Renders the 3D living room scene cycling through LED lighting presets.
Each preset gets its own moment. Run all cells top to bottom.

**Account:** fmmg.mars@gmail.com | **GPU:** T4 (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Cell 1 — GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2 — Install Blender 4.0.2
!wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz
!blender-4.0.2-linux-x64/blender --version

In [ ]:
# Cell 3 — Download the living room scene
# From the cape-jones repo (pushed via PR)
!wget -q https://raw.githubusercontent.com/mhvnsnt/cape-jones/main/3d-assets/living_room.blend -O living_room.blend
!ls -lh living_room.blend

In [ ]:
# Cell 4 — Write the lighting reel render script
%%writefile lighting_reel.py
"""Cape Jones lighting gag reel.
Cycles through LED presets, 3 seconds each. One continuous render.
Run: blender -b living_room.blend --python lighting_reel.py
"""
import bpy, math

# LED presets as (name, [list of RGB keyframes over the 3s segment])
# Each segment = 72 frames @ 24fps
SEG = 72
PRESETS = [
    ("purple-pulse",   [(0.45, 0.10, 0.90), (0.30, 0.05, 0.70), (0.45, 0.10, 0.90)]),
    ("chill-blue",     [(0.10, 0.25, 0.95), (0.05, 0.15, 0.80), (0.10, 0.25, 0.95)]),
    ("ember-glow",     [(0.95, 0.15, 0.10), (0.70, 0.08, 0.05), (0.95, 0.15, 0.10)]),
    ("ghost-mode",     [(0.10, 0.90, 0.85), (0.05, 0.70, 0.65), (0.10, 0.90, 0.85)]),
    ("ocean-calm",     [(0.05, 0.45, 0.85), (0.02, 0.30, 0.65), (0.05, 0.45, 0.85)]),
    ("sunset-cycle",   [(0.95, 0.45, 0.10), (0.90, 0.20, 0.55), (0.95, 0.45, 0.10)]),
    ("rainbow-smooth", [(0.90, 0.10, 0.10), (0.10, 0.90, 0.10), (0.10, 0.10, 0.90), (0.90, 0.10, 0.10)]),
    ("blacklight-flicker", [(0.35, 0.05, 0.80), (0.20, 0.02, 0.50), (0.35, 0.05, 0.80)]),
]

bpy.ops.wm.open_mainfile(filepath="living_room.blend")
scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.device = 'GPU'
scene.cycles.samples = 32  # low samples + denoiser = fast, clean
scene.cycles.use_denoising = True
scene.cycles.denoiser = 'OPTIX'
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.image_settings.codec = 'H264'
scene.render.ffmpeg.format = 'MPEG4'

total = SEG * len(PRESETS)
scene.frame_start = 1
scene.frame_end = total

# Find LED material + room light
led_mat = bpy.data.materials.get("LEDStripMat")
led_node = led_mat.node_tree.nodes["Principled BSDF"]
room_light = bpy.data.objects.get("RoomLight")

# Clear old animation
if led_node.inputs["Emission Color"].id_data.animation_data:
    led_node.inputs["Emission Color"].id_data.animation_data_clear()
if room_light.data.animation_data:
    room_light.data.animation_data_clear()

# Keyframe each preset segment
for i, (name, colors) in enumerate(PRESETS):
    base = i * SEG + 1
    n = len(colors)
    for j, rgb in enumerate(colors):
        f = base + int(j * SEG / max(n - 1, 1))
        led_node.inputs["Emission Color"].default_value = (*rgb, 1)
        led_node.inputs["Emission Color"].keyframe_insert("default_value", frame=f)
        room_light.data.color = rgb
        room_light.data.keyframe_insert("color", frame=f)
    print(f"Preset {i+1}/{len(PRESETS)}: {name} (frames {base}-{base+SEG-1})")

for fc in led_node.inputs["Emission Color"].id_data.animation_data.action.fcurves:
    for kp in fc.keyframe_points:
        kp.interpolation = 'BEZIER'

scene.render.filepath = "/content/cape-jones-lighting-reel.mp4"
bpy.ops.render.render(animation=True)
print(f"DONE: {total} frames, {total/24:.1f}s reel")

In [ ]:
# Cell 5 — Render (this takes a while, ~30-60 min on T4)
!blender-4.0.2-linux-x64/blender -b living_room.blend --python lighting_reel.py 2>&1 | tail -20

In [ ]:
# Cell 6 — Verify + download
!ls -lh /content/cape-jones-lighting-reel.mp4
!ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1 /content/cape-jones-lighting-reel.mp4
from google.colab import files
files.download("/content/cape-jones-lighting-reel.mp4")